# Supplementary Simulation Figure S1 — motif overlap and dynamical mixing

Reproduces the finalized four-panel sensitivity analysis:
- spatial recovery across motif overlap;
- spatial recovery across dynamical mixing;
- temporal recovery across motif overlap;
- temporal recovery across dynamical mixing.

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

RESULTS_CSV = Path(
    "simulation_pca_msaa_extended_outputs/"
    "pca_msaa_extended_results_paper.csv"
)

OUTPUT_DIR = Path("paper_figure_outputs")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

if not RESULTS_CSV.exists():
    raise FileNotFoundError(
        f"Missing {RESULTS_CSV}. Run revision_16_pca_msaa_clean_simulation_benchmark "
        "with RUN_MODE='paper' first."
    )

results = pd.read_csv(RESULTS_CSV)

METHOD_COLORS = {
    "MS-AA": "#1f77b4",
    "PCA": "#ff7f0e",
}

def mean_sem(df, group_col, value_col):
    return (
        df.groupby(group_col)[value_col]
        .agg(["mean", "sem"])
        .reset_index()
    )

FIG_PNG = OUTPUT_DIR / "supp_simulation_overlap_dynamics.png"
FIG_PDF = OUTPUT_DIR / "supp_simulation_overlap_dynamics.pdf"

In [ ]:
overlap = results[
    results["regime"].astype(str) == "metastable_overlap"
].copy()

dynamics = results[
    results["regime"].astype(str) == "metastable_dynamics"
].copy()

overlap_order = [
    "low_overlap",
    "baseline_overlap",
    "high_overlap",
]

dynamics_order = [
    "state_like",
    "baseline",
    "more_mixed",
]

fig, axes = plt.subplots(
    2,
    2,
    figsize=(10.8, 8.0),
    gridspec_kw={"hspace": 0.34, "wspace": 0.28},
)

panels = [
    (
        axes[0, 0],
        overlap,
        overlap_order,
        "condition",
        "aa_motif_recovery",
        "pca_motif_recovery",
        ["Low", "Baseline", "High"],
        "Spatial recovery vs motif overlap",
    ),
    (
        axes[0, 1],
        dynamics,
        dynamics_order,
        "condition",
        "aa_motif_recovery",
        "pca_motif_recovery",
        ["State-like", "Baseline", "More mixed"],
        "Spatial recovery vs dynamical mixing",
    ),
    (
        axes[1, 0],
        overlap,
        overlap_order,
        "condition",
        "aa_temporal_recovery",
        "pca_temporal_recovery",
        ["Low", "Baseline", "High"],
        "Temporal recovery vs motif overlap",
    ),
    (
        axes[1, 1],
        dynamics,
        dynamics_order,
        "condition",
        "aa_temporal_recovery",
        "pca_temporal_recovery",
        ["State-like", "Baseline", "More mixed"],
        "Temporal recovery vs dynamical mixing",
    ),
]

for ax, df, order, group_col, aa_col, pca_col, labels, title in panels:
    x = np.arange(len(order))

    for method, col, shift in [
        ("MS-AA", aa_col, -0.07),
        ("PCA", pca_col, 0.07),
    ]:
        means = []
        sems = []

        for cond in order:
            vals = df.loc[
                df[group_col].astype(str) == cond,
                col,
            ]
            means.append(vals.mean())
            sems.append(vals.sem())

        ax.errorbar(
            x + shift,
            means,
            yerr=sems,
            marker="o",
            linewidth=1.8,
            capsize=3,
            color=METHOD_COLORS[method],
            label=method,
        )

    ax.set_xticks(x)
    ax.set_xticklabels(labels)
    ax.set_ylim(0, 1.05)
    ax.set_ylabel("Recovery (r)")
    ax.set_title(title, fontweight="bold")
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)

for ax, letter in zip(axes.ravel(), list("ABCD")):
    ax.text(
        -0.10, 1.04, letter,
        transform=ax.transAxes,
        fontsize=14,
        fontweight="bold",
        va="top",
    )

axes[0, 1].legend(frameon=False)

fig.suptitle(
    "Sensitivity of spatial and temporal recovery to motif overlap and dynamical mixing",
    fontsize=15,
    fontweight="bold",
)

fig.savefig(FIG_PNG, dpi=300, bbox_inches="tight")
fig.savefig(FIG_PDF, bbox_inches="tight")
plt.show()

print("Saved:", FIG_PNG)
print("Saved:", FIG_PDF)